# prepare_data

Executable notebook module. English explanations precede each functional block. Restart the kernel after changing modules.


In [ ]:
from pathlib import Path
import json, sys, types
PROJECT = Path.cwd()
assert (PROJECT / 'app.ipynb').is_file(), 'Open this notebook from EcoSort-Demo.'

def load_notebook(name):
    if name in sys.modules:
        return sys.modules[name]
    path = PROJECT / (name + '.ipynb')
    notebook = json.loads(path.read_text())
    module = types.ModuleType(name)
    module.__file__ = str(path)
    sys.modules[name] = module
    try:
        for dependency in notebook['metadata'].get('dependencies', []):
            load_notebook(dependency)
        for cell in notebook['cells']:
            if cell['cell_type'] == 'code' and 'module' in cell.get('metadata', {}).get('tags', []):
                exec(compile(''.join(cell['source']), str(path), 'exec'), module.__dict__)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

__file__ = str(PROJECT / 'prepare_data.ipynb')


## Imports and configuration

Load required libraries and define the paths, labels and constants used by the following functions.


In [ ]:
"""Collect sources, map labels, audit images, group duplicates and create an 80/10/10 split.

Restored from the project preparation code. The original 20,000-image selection
is pinned in source_lists/original_20000.csv; it is not resampled on each run.
The source taxonomy is separate from City of Sydney disposal rules.
"""
from concurrent.futures import ThreadPoolExecutor
import hashlib
import json
from pathlib import Path
import random
import zipfile
import argparse
import ssl
import urllib.request
import shutil
import certifi
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from scipy.fft import dctn

BASE = Path(__file__).resolve().parent
DATA = BASE / "data/sources"
LISTS = BASE / "data/source_lists"
OUT = BASE / "data/prepared"

# Public source locations used by the project; upstream downloads may change.
SOURCES = [
    ("TrashNet", "https://raw.githubusercontent.com/garythung/trashnet/master/data/dataset-resized.zip", "TrashNet", None),
    ("RealWaste", "https://archive.ics.uci.edu/static/public/908/realwaste.zip", "RealWaste", None),
    ("household", "https://www.kaggle.com/api/v1/datasets/download/mostafaabla/garbage-classification", "EcoSort_dataset_v3/raw/household", None),
    ("ewaste", "https://www.kaggle.com/api/v1/datasets/download/akshat103/e-waste-image-dataset", "EcoSort_dataset_v3/raw/ewaste", None),
    ("phenom", "https://www.kaggle.com/api/v1/datasets/download/phenomsg/waste-classification", "EcoSort_dataset_v3/raw/phenom", None),
    ("garbage265-train", "https://huggingface.co/datasets/fritz1024/garbage265-classification/resolve/main/train.zip", "garbage265", "e8afee7456b9b449c22dd344839b19afe9208c69bb8ac6518d0280cc1d2f54a1"),
    ("garbage265-val", "https://huggingface.co/datasets/fritz1024/garbage265-classification/resolve/main/val.zip", "garbage265", "6da262bcafeddf87c878bc48e477992efc01369d7f13bc55e4e5a50c28af9c11"),
]




## extract

Validate archive paths before extraction.


In [ ]:
def extract(archive_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError("Unsafe archive path")
        archive.extractall(destination)


## collect_images

Download missing source collections and verify known archive checksums.


In [ ]:
def collect_images():
    """Download only missing source trees; preparation verifies individual images."""
    cache = BASE / "data/downloads"
    context = ssl.create_default_context(cafile=certifi.where())
    for name, url, folder, expected_hash in SOURCES:
        destination = DATA / folder
        check = destination / name.split("-")[-1] if name.startswith("garbage265-") else destination
        if check.is_dir() and any(check.rglob("*.jpg")):
            print("Using existing images:", name, flush=True)
            continue
        cache.mkdir(parents=True, exist_ok=True)
        archive = cache / (name + ".zip")
        if not archive.exists():
            partial = archive.with_suffix(".partial")
            request = urllib.request.Request(url, headers={"User-Agent": "EcoSort coursework"})
            with urllib.request.urlopen(request, context=context, timeout=60) as response, partial.open("wb") as target:
                shutil.copyfileobj(response, target, length=1024*1024)
            partial.replace(archive)
        if expected_hash:
            with archive.open("rb") as stream:
                if hashlib.file_digest(stream, "sha256").hexdigest() != expected_hash:
                    raise ValueError("Archive checksum mismatch: " + name)
        extract(archive, destination)
        print("Collected:", name, flush=True)


## map_label

Translate source taxonomy into the project material labels, independently of disposal rules.


In [ ]:
def map_label(name):
    major, item = name.split("-", 1)  # Map source object names to project classes, not Sydney bin routes.
    if item in {"电池板", "血压计"}:
        return "ewaste"
    if item in {"订书机", "锅盖", "过滤网", "钥匙"}:
        return "mixed_items"
    if item in {"茶壶碎片", "破碎陶瓷", "陶瓷"}:
        return "ceramics"
    if item in {"电池", "纽扣电池", "蓄电池", "充电宝"}:
        return "battery"
    if item in {"药膏", "药片", "药品包装", "药瓶"}:
        return "medicines"
    if item == "灯":
        return "light_bulbs"
    if major == "有害垃圾" or item in {"燃气瓶", "灭火器", "打火机"}:
        return "household_problem_waste"
    if major == "厨余垃圾":
        return "mixed_items" if item in {"秸秆杯", "秸秆碗"} else "food_organics"
    if "玻璃" in item or item in {"酒瓶", "鱼缸"}:
        return "glass"
    if item in {"不锈钢制品", "磁铁", "刀", "钉子", "订书机", "锅", "锅盖", "过滤网", "金属罐", "金属制品", "笼子", "铝制用品", "奶粉罐", "镊子", "铅球", "铁丝球", "钥匙", "图钉", "U型回形针"}:
        return "metal"
    if item in {"保鲜膜内芯", "纸箱"}:
        return "cardboard"
    if item in {"档案袋", "地铁票", "登机牌", "吊牌", "购物纸袋", "卡片", "快递纸袋", "日历", "书", "信封", "纸牌", "纸制品", "便利贴", "彩票", "车票", "电影票"}:
        return "paper"
    if item in {"果冻杯", "泡沫板", "食用油桶", "塑料碗盆", "塑料制品", "洗发水瓶", "瑜伽球", "塑料袋"}:
        return "plastic"
    if item in {"布制品", "耳套", "裤子", "帽子", "裙子", "毯子", "袜子", "衣服", "枕头", "毛巾", "厨房抹布", "眼镜布"}:
        return "textile_trash"
    if item in {"木雕", "木棍", "木桶", "木制切菜板", "木质锅铲", "木质梳子", "串串竹签", "牙签", "竹筷"}:
        return "wood"
    if item in {"插头电线", "插线板", "充电头", "充电线", "充电牙刷", "吹风机", "打印机", "电磁炉", "电动卷发棒", "电动剃须刀", "电饭煲", "电风扇", "电话", "电路板", "显示器", "电视机", "电熨斗", "电子秤", "豆浆机", "耳机", "话筒", "计算器", "键盘", "空气加湿器", "空气净化器", "路由器", "闹铃", "燃气灶", "扫地机器人", "收音机", "手表", "手电筒", "手机", "台灯", "太阳能热水器", "体重秤", "网卡", "遥控器", "音响", "电蚊香"}:
        return "ewaste"
    if major == "其他垃圾":
        return "trash"
    return "mixed_items"


## audit

Check image quality and calculate file, pixel and perceptual fingerprints.


In [ ]:
def audit(row):
    path = Path(row["path"])
    try:
        content = path.read_bytes()
        with Image.open(path) as source:
            image = ImageOps.exif_transpose(source).convert("RGB")
            image.load()
            if min(image.size) < 64:  # Reject images too small to retain useful visual detail.
                return {**row, "status": "too_small"}
            pixels = np.asarray(image.resize((128, 128)))
            gray = image.convert("L").resize((32, 32))
            variants = []
            for transform in (None, Image.Transpose.ROTATE_90, Image.Transpose.ROTATE_180, Image.Transpose.ROTATE_270):
                rotated = gray if transform is None else gray.transpose(transform)
                for candidate in (rotated, ImageOps.mirror(rotated)):
                    values = dctn(np.asarray(candidate, dtype=float), norm="ortho")[:8, :8].ravel()[1:]
                    variants.append(int.from_bytes(np.packbits(values > np.median(values)).tobytes(), "big"))
        return {**row, "sha256": hashlib.sha256(content).hexdigest(), "pixel_hash": hashlib.sha256(pixels.tobytes()).hexdigest(), "phash": str(min(variants)), "status": "valid"}
    except Exception as error:
        return {**row, "status": "invalid", "error": str(error)}


## main

Execute this module workflow. Read the ordered steps and parameters below before enabling execution.


In [ ]:
def main():
    global OUT
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--download", action="store_true", help="Collect missing source images first.")
    parser.add_argument("--output", type=Path, default=OUT, help="A new directory for reconstructed splits.")
    args = parser.parse_args()
    OUT = args.output.resolve()
    OUT.mkdir(parents=True, exist_ok=False)  # Never overwrite the split used by the deployed model.
    if args.download:
        collect_images()
    names = (LISTS / "classname.txt").read_text().splitlines()
    mapping = {index: {"source_label": name, "label": map_label(name)} for index, name in enumerate(names)}
    (OUT / "category_mapping.json").write_text(json.dumps(mapping, ensure_ascii=False, indent=2))
    rows = []
    old = pd.read_csv(LISTS / "original_20000.csv")
    assert len(old) == 20000
    for row in old.itertuples():
        path = (BASE / row.path).resolve()
        assert path.is_relative_to(DATA)
        assert path.is_file(), path
        assert hashlib.sha256(path.read_bytes()).hexdigest() == row.sha256
        rows.append({"dataset": row.dataset, "label": row.label, "fine_label": row.label, "path": str(path), "family": ""})
    for filename in ("train.csv", "val.csv"):
        frame = pd.read_csv(LISTS / filename)
        for relative, index in frame.itertuples(index=False, name=None):
            path = DATA / "garbage265" / relative
            assert path.is_file(), path
            label = mapping[int(index)]
            rows.append({"dataset": "garbage265", "label": label["label"], "fine_label": label["source_label"], "path": str(path), "family": "garbage265:" + path.stem.split("_")[0]})
    print("Raw candidates:", len(rows), flush=True)
    # Decode and hash original files again instead of trusting a previous audit cache.
    audited = []
    with ThreadPoolExecutor(max_workers=10) as pool:
        for result in pool.map(audit, rows):
            audited.append(result)
            if len(audited) % 10000 == 0:
                print("Audited:", len(audited), flush=True)
    frame = pd.DataFrame(audited)
    current_labels = {row["path"]: row["label"] for row in rows}
    frame["label"] = frame.path.map(current_labels)
    rejected = {
        "RealWaste/realwaste-main/RealWaste/Cardboard/Cardboard_414.jpg": "Crushed beverage can, not cardboard",
        "RealWaste/realwaste-main/RealWaste/Cardboard/Cardboard_364.jpg": "Crushed beverage can, not cardboard",
        "RealWaste/realwaste-main/RealWaste/Cardboard/Cardboard_249.jpg": "Foil wrapper, not cardboard",
        "garbage265/train/146/c873832ee9ee_50.jpg": "Street scene, not a visible plastic item",
    }
    (OUT / "manual_rejections.json").write_text(json.dumps(rejected, indent=2))
    frame.loc[frame.path.isin([str(DATA / path) for path in rejected]), "status"] = "manual_reject"
    valid = frame[frame.status == "valid"].reset_index(drop=True)
    parent = list(range(len(valid)))
    def find(index):
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index
    def merge(a, b):
        a, b = find(a), find(b)
        if a != b:
            parent[b] = a
    for key in ("sha256", "pixel_hash", "family"):  # Keep matching files, pixels and source families in one group.
        for value, indexes in valid.groupby(key).groups.items():
            if key == "family" and (not value or pd.isna(value)):
                continue
            indexes = list(indexes)
            for index in indexes[1:]:
                merge(indexes[0], index)
    buckets = {}
    hashes = [int(value) for value in valid.phash]
    for index, value in enumerate(hashes):
        candidates = set()
        for band in range(3):
            key = (band, (value >> (band * 22)) & ((1 << 22) - 1))
            candidates.update(buckets.get(key, [])[-300:])
        for other in candidates:
            if (value ^ hashes[other]).bit_count() <= 2:  # Heuristic visual similarity; not a proof of duplicate identity.
                merge(index, other)
        for band in range(3):
            key = (band, (value >> (band * 22)) & ((1 << 22) - 1))
            buckets.setdefault(key, []).append(index)
    valid["group_id"] = [find(i) for i in range(len(valid))]
    conflicts = valid.groupby("group_id").label.nunique()
    conflict_ids = set(conflicts[conflicts > 1].index)  # Exclude whole groups whose labels disagree.
    excluded = valid[valid.group_id.isin(conflict_ids)].copy()
    excluded["path"] = excluded.path.map(lambda p: str(Path(p).relative_to(BASE)))
    excluded.to_csv(OUT / "conflicting_labels_excluded.csv", index=False)
    clean = valid[~valid.group_id.isin(conflict_ids)].drop_duplicates("sha256").drop_duplicates("pixel_hash").copy()
    rng = random.Random(42)
    assignments = {}
    for label, group in clean.groupby("label", sort=True):
        ids = sorted(group.group_id.unique())
        rng.shuffle(ids)
        a, b = round(len(ids) * 0.8), round(len(ids) * 0.9)  # Split groups 80/10/10; never separate related images.
        for split, subset in (("train", ids[:a]), ("validation", ids[a:b]), ("test", ids[b:])):
            assignments.update({identifier: split for identifier in subset})
    clean["split"] = clean.group_id.map(assignments)
    assert clean.sha256.is_unique and clean.pixel_hash.is_unique
    assert clean.groupby("group_id").split.nunique().max() == 1
    clean["path"] = clean.path.map(lambda p: str(Path(p).relative_to(BASE)))
    clean.to_csv(OUT / "split_manifest.csv", index=False)
    labels = sorted(clean.label.unique())
    (OUT / "class_names.json").write_text(json.dumps(labels, indent=2))
    counts = pd.crosstab(clean.label, clean.split)
    counts.to_csv(OUT / "class_counts.csv")
    report = {"raw_candidate_images": len(rows), "garbage265_metadata_images": len(rows) - 20000, "original_images": 20000, "valid_decoded": len(valid), "retained_images": len(clean), "split_counts": clean.groupby("split").size().to_dict(), "class_counts": clean.groupby("label").size().to_dict(), "conflicting_groups_removed": len(conflict_ids), "duplicate_group_cross_split": False, "note": "Source taxonomies are retained as fine_label. Chinese disposal categories are not used as Sydney disposal rules. Near-duplicate filtering is heuristic, not a proof of independence."}
    (OUT / "dataset_verification.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2), flush=True)


## Optional execution

Disabled by default to avoid starting expensive work or overwriting outputs during inspection.


In [ ]:
RUN_WORKFLOW = False
if RUN_WORKFLOW:
    saved_arguments = sys.argv
    try:
        sys.argv = ['prepare_data']
        main()
    finally:
        sys.argv = saved_arguments
